In [7]:
import re
import os
import pandas as pd

def TIC_RSD(intensities):
    """
    Calculate the Relative Standard Deviation (RSD) for a given intensity array.

    Args:
        intensities (list of int): The intensity values.

    Returns:
        float: The RSD percentage.
    """
    if not intensities:
        return 0.0
    mean = sum(intensities) / len(intensities)
    if mean == 0:
        return 0.0
    variance = sum((x - mean) ** 2 for x in intensities) / len(intensities)
    std_dev = variance ** 0.5
    rsd = (std_dev / mean) * 100
    return rsd

def TIC_Summed(intensities):
    """
    Calculate the summed Total Ion Current (TIC) for a given intensity array.

    Args:
        intensities (list of int): The intensity values.

    Returns:
        int: The summed TIC.
    """
    return sum(intensities) if intensities else 0

def parse_chromatogram_data(input_dir, output_dir):
    """
    Parse chromatogram data from text files in the specified input directory
    and save the results as a CSV in the specified output directory.

    Args:
        input_dir (str): Directory containing the input text files.
        output_dir (str): Directory to save the output CSV file.

    Returns:
        pd.DataFrame: DataFrame containing the parsed chromatogram data.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, 'parsed_chromatogram_data.csv')

    # Initialize lists to store parsed data
    filenames = []
    q1_values = []
    q3_values = []
    lipids = []
    dates = []          # List for Date
    sample_names = []   # List for Sample_Name
    samples = []        # List for Sample
    summed_intensities = []  # List for Summed_Intensity
    tic_rsd_values = [] # List for TIC_RSD
    tic_summed_values = [] # List for TIC_Summed

    # Iterate through all .txt files in the specified directory
    for file_name in os.listdir(input_dir):
        if file_name.endswith('.txt'):
            file_path = os.path.join(input_dir, file_name)

            # Extract Date and Sample_Name from the filename
            base_name = os.path.splitext(file_name)[0]  # Removes the .txt extension
            parts = base_name.split('_', 1)             # Split only on the first underscore
            if len(parts) == 2:
                date_str = parts[0]                     # e.g., '20241115'
                sample_name = parts[1]                  # e.g., 'Plasma_Acyl-Carnitines'
            else:
                # Handle unexpected filename formats
                date_str = ''
                sample_name = ''

            # Determine the Sample value based on Sample_Name
            sample = 'Blank' if 'Blank' in sample_name else 'Sample'

            # Open and read all lines of the file
            with open(file_path, 'r') as file:
                lines = file.readlines()

            # -------------------
            # Extract TIC Data
            # -------------------
            TIC_intensities = []
            tic_found = False  # Flag to indicate if TIC section is found
            in_tic_section = False  # Flag to indicate parsing within TIC
            for i, line in enumerate(lines):
                if 'id: TIC' in line:
                    tic_found = True
                    in_tic_section = True
                    print(f"[DEBUG] Found 'id: TIC' in file: {file_name} at line {i}")
                    continue  # Move to the next line

                if in_tic_section:
                    # Look for 'cvParam: intensity array, number of detector counts'
                    if 'cvParam: intensity array' in line:
                        print(f"[DEBUG] Found 'cvParam: intensity array' in file: {file_name} at line {i}")
                        # The intensity values are expected in the next line
                        if i + 1 < len(lines):
                            intensity_line = lines[i + 1].strip()
                            print(f"[DEBUG] Intensity line for TIC in file {file_name}: {intensity_line}")
                            match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', intensity_line)
                            if match:
                                TIC_intensities = list(map(int, match.group(1).split()))
                                print(f"[DEBUG] Parsed TIC intensities for file {file_name}: {TIC_intensities}")
                            else:
                                print(f"[ERROR] Failed to parse TIC intensities in file {file_name} at line {i + 1}")
                        else:
                            print(f"[ERROR] Intensity line missing after 'cvParam: intensity array' in file {file_name}")
                        break  # Exit after finding the intensity array

            if not tic_found:
                print(f"[WARNING] No 'id: TIC' section found in file: {file_name}")

            # Calculate TIC_RSD using the TIC_RSD function
            TIC_rsd = TIC_RSD(TIC_intensities)
            print(f"[DEBUG] Calculated TIC_RSD for file {file_name}: {TIC_rsd}")

            # Calculate TIC_Summed using the TIC_Summed function
            TIC_sum = TIC_Summed(TIC_intensities)
            print(f"[DEBUG] Calculated TIC_Summed for file {file_name}: {TIC_sum}")

            # -------------------
            # Parse Lipid Data
            # -------------------
            current_filename = ""
            current_q1 = None
            current_q3 = None
            current_lipid = ""
            parsing_intensity = False
            intensities = []

            for line in lines:
                # Extract the filename (assumes filename appears earlier in the file)
                if 'sourceFile:' in line or 'name:' in line:
                    match = re.search(r'name:\s+([\w.]+)', line)
                    if match:
                        current_filename = match.group(1)
                        print(f"[DEBUG] Extracted current_filename: {current_filename}")

                # Extract Q1, Q3 values, and Lipid name
                if 'id: SRM SIC Q1=' in line:
                    match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name=([^\s]+)', line)
                    if match:
                        current_q1 = float(match.group(1))
                        current_q3 = float(match.group(2))
                        current_lipid = match.group(3)
                        print(f"[DEBUG] Extracted Lipid data - Q1: {current_q1}, Q3: {current_q3}, Lipid: {current_lipid}")

                # Check if we are parsing intensity array data for lipids
                if 'cvParam: intensity array' in line:
                    parsing_intensity = True
                    intensities = []
                    print(f"[DEBUG] Start parsing intensity array for lipid in file {file_name}")
                elif parsing_intensity and 'binary: [' in line:
                    # Extract intensity values
                    match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
                    if match:
                        intensities = list(map(int, match.group(1).split()))
                        current_intensity_sum = sum(intensities)
                        print(f"[DEBUG] Parsed lipid intensities: {intensities}")
                        print(f"[DEBUG] Summed intensity for lipid: {current_intensity_sum}")

                        # Append the extracted and calculated data to the lists
                        if current_filename and current_q1 is not None and current_q3 is not None:
                            filenames.append(current_filename)
                            q1_values.append(current_q1)
                            q3_values.append(current_q3)
                            lipids.append(current_lipid)
                            dates.append(date_str)              # Append Date
                            sample_names.append(sample_name)    # Append Sample_Name
                            samples.append(sample)              # Append Sample
                            summed_intensities.append(current_intensity_sum)  # Append Summed_Intensity
                            tic_rsd_values.append(TIC_rsd)      # Append TIC_RSD
                            tic_summed_values.append(TIC_sum)   # Append TIC_Summed
                            print(f"[DEBUG] Appended data for lipid: {current_lipid} in file {file_name}")

                    else:
                        print(f"[ERROR] Failed to parse lipid intensities in file {file_name}")

                    parsing_intensity = False

    # Create a DataFrame
    chromatogram_df = pd.DataFrame({
        'Date': dates,                        # Date column
        'Sample_Name': sample_names,          # Sample_Name column
        'Sample': samples,                    # Sample column
        'Lipid': lipids,
        'Q1': q1_values,
        'Q3': q3_values,
        'Summed_Intensity': summed_intensities,  # Summed_Intensity column
        'TIC_RSD': tic_rsd_values,            # TIC_RSD column
        'TIC_Summed': tic_summed_values,      # TIC_Summed column
        'Filename': filenames,
    })

    # Create Base_Sample_Name by removing 'Blank_' prefix if present
    chromatogram_df['Base_Sample_Name'] = chromatogram_df['Sample_Name'].str.replace('Blank_', '', regex=False)

    # Assign group numbers based on Base_Sample_Name
    chromatogram_df['Blank_Group'] = pd.factorize(chromatogram_df['Base_Sample_Name'])[0] + 1  # Start groups at 1

    # Optionally, drop the Base_Sample_Name column if not needed
    chromatogram_df.drop(columns=['Base_Sample_Name'], inplace=True)

    # Optionally, convert Date to datetime format
    # chromatogram_df['Date'] = pd.to_datetime(chromatogram_df['Date'], format='%Y%m%d')

    # Reorder columns for better readability
    columns_order = [
        'Date', 
        'Sample_Name', 
        'Sample', 
        'Blank_Group', 
        'Lipid', 
        'Q1', 
        'Q3', 
        'Summed_Intensity',
        'TIC_RSD',                           # Include TIC_RSD in the order
        'TIC_Summed',                       # Include TIC_Summed in the order
        'Filename'
    ]
    chromatogram_df = chromatogram_df[columns_order]

    # Save the DataFrame to a CSV file
    chromatogram_df.to_csv(output_file, index=False)
    print(f"[INFO] Successfully saved parsed data to {output_file}")

    return chromatogram_df


In [9]:
import re
import os
import pandas as pd

class QTRAP_Parse:
    def __init__(self, input_dir, output_dir):
        """
        Initialize the QTRAP_Parse class with input and output directories.

        Args:
            input_dir (str): Directory containing the input text files.
            output_dir (str): Directory to save the output CSV file.
        """
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.output_file = os.path.join(self.output_dir, 'parsed_chromatogram_data.csv')

        # Initialize lists to store parsed data
        self.filenames = []
        self.q1_values = []
        self.q3_values = []
        self.lipids = []
        self.dates = []          # List for Date
        self.sample_names = []   # List for Sample_Name
        self.samples = []        # List for Sample
        self.summed_intensities = []  # List for Summed_Intensity
        self.tic_rsd_values = [] # List for TIC_RSD
        self.tic_summed_values = [] # List for TIC_Summed

    def TIC_RSD(self, intensities):
        """
        Calculate the Relative Standard Deviation (RSD) for a given intensity array.

        Args:
            intensities (list of int): The intensity values.

        Returns:
            float: The RSD percentage.
        """
        if not intensities:
            return 0.0
        mean = sum(intensities) / len(intensities)
        if mean == 0:
            return 0.0
        variance = sum((x - mean) ** 2 for x in intensities) / len(intensities)
        std_dev = variance ** 0.5
        rsd = (std_dev / mean) * 100
        return rsd

    def TIC_Summed(self, intensities):
        """
        Calculate the summed Total Ion Current (TIC) for a given intensity array.

        Args:
            intensities (list of int): The intensity values.

        Returns:
            int: The summed TIC.
        """
        return sum(intensities) if intensities else 0

    def parse_file(self, file_name):
        """
        Parse a single chromatogram text file to extract necessary data.

        Args:
            file_name (str): The name of the file to parse.
        """
        file_path = os.path.join(self.input_dir, file_name)

        # Extract Date and Sample_Name from the filename
        base_name = os.path.splitext(file_name)[0]  # Removes the .txt extension
        parts = base_name.split('_', 1)             # Split only on the first underscore
        if len(parts) == 2:
            date_str = parts[0]                     # e.g., '20241115'
            sample_name = parts[1]                  # e.g., 'Plasma_Acyl-Carnitines'
        else:
            # Handle unexpected filename formats
            date_str = ''
            sample_name = ''

        # Determine the Sample value based on Sample_Name
        sample = 'Blank' if 'Blank' in sample_name else 'Sample'

        # Open and read all lines of the file
        try:
            with open(file_path, 'r') as file:
                lines = file.readlines()
        except Exception as e:
            print(f"[ERROR] Failed to read file {file_name}: {e}")
            return  # Skip to the next file

        # -------------------
        # Extract TIC Data
        # -------------------
        TIC_intensities = []
        tic_found = False  # Flag to indicate if TIC section is found
        in_tic_section = False  # Flag to indicate parsing within TIC

        for i, line in enumerate(lines):
            if 'id: TIC' in line:
                tic_found = True
                in_tic_section = True
                print(f"[DEBUG] Found 'id: TIC' in file: {file_name} at line {i}")
                continue  # Move to the next line

            if in_tic_section:
                # Look for 'cvParam: intensity array, number of detector counts'
                if 'cvParam: intensity array' in line:
                    print(f"[DEBUG] Found 'cvParam: intensity array' in file: {file_name} at line {i}")
                    # The intensity values are expected in the next line
                    if i + 1 < len(lines):
                        intensity_line = lines[i + 1].strip()
                        print(f"[DEBUG] Intensity line for TIC in file {file_name}: {intensity_line}")
                        match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', intensity_line)
                        if match:
                            try:
                                TIC_intensities = list(map(int, match.group(1).split()))
                                print(f"[DEBUG] Parsed TIC intensities for file {file_name}: {TIC_intensities}")
                            except ValueError as ve:
                                print(f"[ERROR] Non-integer value found in TIC intensities in file {file_name}: {ve}")
                        else:
                            print(f"[ERROR] Failed to parse TIC intensities in file {file_name} at line {i + 1}")
                    else:
                        print(f"[ERROR] Intensity line missing after 'cvParam: intensity array' in file {file_name}")
                    break  # Exit after finding the intensity array

        if not tic_found:
            print(f"[WARNING] No 'id: TIC' section found in file: {file_name}")

        # Calculate TIC_RSD using the TIC_RSD function
        TIC_rsd = self.TIC_RSD(TIC_intensities)
        print(f"[DEBUG] Calculated TIC_RSD for file {file_name}: {TIC_rsd}")

        # Calculate TIC_Summed using the TIC_Summed function
        TIC_sum = self.TIC_Summed(TIC_intensities)
        print(f"[DEBUG] Calculated TIC_Summed for file {file_name}: {TIC_sum}")

        # -------------------
        # Parse Lipid Data
        # -------------------
        current_filename = ""
        current_q1 = None
        current_q3 = None
        current_lipid = ""
        parsing_intensity = False
        intensities = []

        for i, line in enumerate(lines):
            # Extract the filename (assumes filename appears earlier in the file)
            if 'sourceFile:' in line or 'name:' in line:
                match = re.search(r'name:\s+([\w.]+)', line)
                if match:
                    current_filename = match.group(1)
                    print(f"[DEBUG] Extracted current_filename: {current_filename}")

            # Extract Q1, Q3 values, and Lipid name
            if 'id: SRM SIC Q1=' in line:
                match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name=([^\s]+)', line)
                if match:
                    current_q1 = float(match.group(1))
                    current_q3 = float(match.group(2))
                    current_lipid = match.group(3)
                    print(f"[DEBUG] Extracted Lipid data - Q1: {current_q1}, Q3: {current_q3}, Lipid: {current_lipid}")

            # Check if we are parsing intensity array data for lipids
            if 'cvParam: intensity array' in line:
                parsing_intensity = True
                intensities = []
                print(f"[DEBUG] Start parsing intensity array for lipid in file {file_name}")
            elif parsing_intensity and 'binary: [' in line:
                # Extract intensity values
                match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
                if match:
                    try:
                        intensities = list(map(int, match.group(1).split()))
                        current_intensity_sum = sum(intensities)
                        print(f"[DEBUG] Parsed lipid intensities: {intensities}")
                        print(f"[DEBUG] Summed intensity for lipid: {current_intensity_sum}")
                    except ValueError as ve:
                        print(f"[ERROR] Non-integer value found in lipid intensities in file {file_name}: {ve}")
                        intensities = []
                        current_intensity_sum = 0

                    # Append the extracted and calculated data to the lists
                    if current_filename and current_q1 is not None and current_q3 is not None:
                        self.filenames.append(current_filename)
                        self.q1_values.append(current_q1)
                        self.q3_values.append(current_q3)
                        self.lipids.append(current_lipid)
                        self.dates.append(date_str)              # Append Date
                        self.sample_names.append(sample_name)    # Append Sample_Name
                        self.samples.append(sample)              # Append Sample
                        self.summed_intensities.append(current_intensity_sum)  # Append Summed_Intensity
                        self.tic_rsd_values.append(TIC_rsd)      # Append TIC_RSD
                        self.tic_summed_values.append(TIC_sum)   # Append TIC_Summed
                        print(f"[DEBUG] Appended data for lipid: {current_lipid} in file {file_name}")

                else:
                    print(f"[ERROR] Failed to parse lipid intensities in file {file_name}")
                parsing_intensity = False

    def parse_all_files(self):
        """
        Iterate through all .txt files in the input directory and parse them.
        """
        if not os.path.isdir(self.input_dir):
            print(f"[ERROR] Input directory '{self.input_dir}' does not exist.")
            return

        # Ensure the output directory exists
        os.makedirs(self.output_dir, exist_ok=True)

        # Iterate through all .txt files in the specified directory
        for file_name in os.listdir(self.input_dir):
            if file_name.endswith('.txt'):
                print(f"[INFO] Processing file: {file_name}")
                self.parse_file(file_name)

    def generate_dataframe(self):
        """
        Generate a Pandas DataFrame from the parsed data.

        Returns:
            pd.DataFrame: DataFrame containing the parsed chromatogram data.
        """
        chromatogram_df = pd.DataFrame({
            'Date': self.dates,                        # Date column
            'Sample_Name': self.sample_names,          # Sample_Name column
            'Sample': self.samples,                    # Sample column
            'Lipid': self.lipids,
            'Q1': self.q1_values,
            'Q3': self.q3_values,
            'Summed_Intensity': self.summed_intensities,  # Summed_Intensity column
            'TIC_RSD': self.tic_rsd_values,            # TIC_RSD column
            'TIC_Summed': self.tic_summed_values,      # TIC_Summed column
            'Filename': self.filenames,
        })

        # Create Base_Sample_Name by removing 'Blank_' prefix if present
        chromatogram_df['Base_Sample_Name'] = chromatogram_df['Sample_Name'].str.replace('Blank_', '', regex=False)

        # Assign group numbers based on Base_Sample_Name
        chromatogram_df['Blank_Group'] = pd.factorize(chromatogram_df['Base_Sample_Name'])[0] + 1  # Start groups at 1

        # Optionally, drop the Base_Sample_Name column if not needed
        chromatogram_df.drop(columns=['Base_Sample_Name'], inplace=True)

        # Optionally, convert Date to datetime format
        # chromatogram_df['Date'] = pd.to_datetime(chromatogram_df['Date'], format='%Y%m%d')

        # Reorder columns for better readability
        columns_order = [
            'Date', 
            'Sample_Name', 
            'Sample', 
            'Blank_Group', 
            'Lipid', 
            'Q1', 
            'Q3', 
            'Summed_Intensity',
            'TIC_RSD',                           # Include TIC_RSD in the order
            'TIC_Summed',                       # Include TIC_Summed in the order
            'Filename'
        ]
        chromatogram_df = chromatogram_df[columns_order]

        return chromatogram_df

    def save_to_csv(self, dataframe):
        """
        Save the DataFrame to a CSV file.

        Args:
            dataframe (pd.DataFrame): The DataFrame to save.
        """
        try:
            dataframe.to_csv(self.output_file, index=False)
            print(f"[INFO] Successfully saved parsed data to {self.output_file}")
        except Exception as e:
            print(f"[ERROR] Failed to save DataFrame to CSV: {e}")

    def run(self):
        """
        Execute the parsing process: parse all files, generate DataFrame, and save to CSV.
        """
        self.parse_all_files()
        df = self.generate_dataframe()
        self.save_to_csv(df)
        return df


In [13]:
import re
import os
import pandas as pd

class QTRAP_Parse:
    def __init__(self, input_dir, output_dir):
        """
        Initialize the QTRAP_Parse class with input and output directories.

        Args:
            input_dir (str): Directory containing the input text files.
            output_dir (str): Directory to save the output CSV file.
        """
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.output_file = os.path.join(self.output_dir, 'parsed_chromatogram_data.csv')

        # Initialize lists to store parsed data
        self.filenames = []
        self.q1_values = []
        self.q3_values = []
        self.lipids = []
        self.dates = []          # List for Date
        self.sample_names = []   # List for Sample_Name
        self.samples = []        # List for Sample
        self.summed_intensities = []  # List for Summed_Intensity
        self.tic_rsd_values = [] # List for TIC_RSD
        self.tic_summed_values = [] # List for TIC_Summed

    def TIC_RSD(self, intensities):
        """
        Calculate the Relative Standard Deviation (RSD) for a given intensity array.

        Args:
            intensities (list of int): The intensity values.

        Returns:
            float: The RSD percentage rounded to two decimal places.
        """
        if not intensities:
            return 0.0
        mean = sum(intensities) / len(intensities)
        if mean == 0:
            return 0.0
        variance = sum((x - mean) ** 2 for x in intensities) / len(intensities)
        std_dev = variance ** 0.5
        rsd = (std_dev / mean) * 100
        return round(rsd, 2)

    def TIC_Summed(self, intensities):
        """
        Calculate the summed Total Ion Current (TIC) for a given intensity array.

        Args:
            intensities (list of int): The intensity values.

        Returns:
            int: The summed TIC.
        """
        return sum(intensities) if intensities else 0

    def parse_file(self, file_name):
        """
        Parse a single chromatogram text file to extract necessary data.

        Args:
            file_name (str): The name of the file to parse.
        """
        file_path = os.path.join(self.input_dir, file_name)

        # Extract Date and Sample_Name from the filename
        base_name = os.path.splitext(file_name)[0]  # Removes the .txt extension
        parts = base_name.split('_', 1)             # Split only on the first underscore
        if len(parts) == 2:
            date_str = parts[0]                     # e.g., '20241115'
            sample_name = parts[1]                  # e.g., 'Plasma_Acyl-Carnitines'
        else:
            # Handle unexpected filename formats
            date_str = ''
            sample_name = ''

        # Determine the Sample value based on Sample_Name
        sample = 'Blank' if 'Blank' in sample_name else 'Sample'

        # Open and read all lines of the file
        try:
            with open(file_path, 'r') as file:
                lines = file.readlines()
        except Exception:
            # Skip the file if it cannot be read
            return

        # -------------------
        # Extract TIC Data
        # -------------------
        TIC_intensities = []
        tic_found = False  # Flag to indicate if TIC section is found
        in_tic_section = False  # Flag to indicate parsing within TIC

        for i, line in enumerate(lines):
            if 'id: TIC' in line:
                tic_found = True
                in_tic_section = True
                continue  # Move to the next line

            if in_tic_section:
                # Look for 'cvParam: intensity array, number of detector counts'
                if 'cvParam: intensity array' in line:
                    # The intensity values are expected in the next line
                    if i + 1 < len(lines):
                        intensity_line = lines[i + 1].strip()
                        match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', intensity_line)
                        if match:
                            try:
                                TIC_intensities = list(map(int, match.group(1).split()))
                            except ValueError:
                                TIC_intensities = []
                        # Exit after finding the intensity array
                    break

        # Calculate TIC_RSD using the TIC_RSD function
        TIC_rsd = self.TIC_RSD(TIC_intensities)

        # Calculate TIC_Summed using the TIC_Summed function
        TIC_sum = self.TIC_Summed(TIC_intensities)

        # -------------------
        # Parse Lipid Data
        # -------------------
        current_filename = ""
        current_q1 = None
        current_q3 = None
        current_lipid = ""
        parsing_intensity = False
        intensities = []

        for line in lines:
            # Extract the filename (assumes filename appears earlier in the file)
            if 'sourceFile:' in line or 'name:' in line:
                match = re.search(r'name:\s+([\w.]+)', line)
                if match:
                    current_filename = match.group(1)

            # Extract Q1, Q3 values, and Lipid name
            if 'id: SRM SIC Q1=' in line:
                match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name=([^\s]+)', line)
                if match:
                    current_q1 = float(match.group(1))
                    current_q3 = float(match.group(2))
                    current_lipid = match.group(3)

            # Check if we are parsing intensity array data for lipids
            if 'cvParam: intensity array' in line:
                parsing_intensity = True
                intensities = []
            elif parsing_intensity and 'binary: [' in line:
                # Extract intensity values
                match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
                if match:
                    try:
                        intensities = list(map(int, match.group(1).split()))
                        current_intensity_sum = sum(intensities)
                    except ValueError:
                        intensities = []
                        current_intensity_sum = 0

                    # Append the extracted and calculated data to the lists
                    if current_filename and current_q1 is not None and current_q3 is not None:
                        self.filenames.append(current_filename)
                        self.q1_values.append(current_q1)
                        self.q3_values.append(current_q3)
                        self.lipids.append(current_lipid)
                        self.dates.append(date_str)              # Append Date
                        self.sample_names.append(sample_name)    # Append Sample_Name
                        self.samples.append(sample)              # Append Sample
                        self.summed_intensities.append(current_intensity_sum)  # Append Summed_Intensity
                        self.tic_rsd_values.append(TIC_rsd)      # Append TIC_RSD
                        self.tic_summed_values.append(TIC_sum)   # Append TIC_Summed

                parsing_intensity = False

    def parse_all_files(self):
        """
        Iterate through all .txt files in the input directory and parse them.
        """
        if not os.path.isdir(self.input_dir):
            return

        # Ensure the output directory exists
        os.makedirs(self.output_dir, exist_ok=True)

        # Iterate through all .txt files in the specified directory
        for file_name in os.listdir(self.input_dir):
            if file_name.endswith('.txt'):
                self.parse_file(file_name)

    def generate_dataframe(self):
        """
        Generate a Pandas DataFrame from the parsed data.

        Returns:
            pd.DataFrame: DataFrame containing the parsed chromatogram data.
        """
        chromatogram_df = pd.DataFrame({
            'Date': self.dates,                        # Date column
            'Sample_Name': self.sample_names,          # Sample_Name column
            'Sample': self.samples,                    # Sample column
            'Lipid': self.lipids,
            'Q1': self.q1_values,
            'Q3': self.q3_values,
            'Summed_Intensity': self.summed_intensities,  # Summed_Intensity column
            'TIC_RSD': self.tic_rsd_values,            # TIC_RSD column
            'TIC_Summed': self.tic_summed_values,      # TIC_Summed column
            'Filename': self.filenames,
        })

        # Create Base_Sample_Name by removing 'Blank_' prefix if present
        chromatogram_df['Base_Sample_Name'] = chromatogram_df['Sample_Name'].str.replace('Blank_', '', regex=False)

        # Assign group numbers based on Base_Sample_Name
        chromatogram_df['Blank_Group'] = pd.factorize(chromatogram_df['Base_Sample_Name'])[0] + 1  # Start groups at 1

        # Optionally, drop the Base_Sample_Name column if not needed
        chromatogram_df.drop(columns=['Base_Sample_Name'], inplace=True)

        # Optionally, convert Date to datetime format
        # chromatogram_df['Date'] = pd.to_datetime(chromatogram_df['Date'], format='%Y%m%d')

        # Reorder columns for better readability
        columns_order = [
            'Date', 
            'Sample_Name', 
            'Sample', 
            'Blank_Group', 
            'Lipid', 
            'Q1', 
            'Q3', 
            'Summed_Intensity',
            'TIC_RSD',                           # Include TIC_RSD in the order
            'TIC_Summed',                       # Include TIC_Summed in the order
            'Filename'
        ]
        chromatogram_df = chromatogram_df[columns_order]

        return chromatogram_df

    def save_to_csv(self, dataframe):
        """
        Save the DataFrame to a CSV file.

        Args:
            dataframe (pd.DataFrame): The DataFrame to save.
        """
        try:
            dataframe.to_csv(self.output_file, index=False)
        except Exception:
            # Handle exceptions silently or implement alternative handling if needed
            pass

    def run(self):
        """
        Execute the parsing process: parse all files, generate DataFrame, and save to CSV.

        Returns:
            pd.DataFrame: The resulting DataFrame after parsing.
        """
        self.parse_all_files()
        df = self.generate_dataframe()
        self.save_to_csv(df)
        return df


In [14]:
# # Import the function from QTRAP.py
# from QTRAP import parse_chromatogram_data

# Define input and output directories
input_dir = 'data/text/'
output_dir = 'result/'

# Parse the chromatogram data and save to a CSV
chromatogram_df = parse_chromatogram_data(input_dir, output_dir)

# Display the resulting DataFrame
chromatogram_df


[DEBUG] Found 'id: TIC' in file: 20241115_Plasma_TG_22-6.txt at line 90
[DEBUG] Found 'cvParam: intensity array' in file: 20241115_Plasma_TG_22-6.txt at line 97
[DEBUG] Intensity line for TIC in file 20241115_Plasma_TG_22-6.txt: binary: [60] 2040 1440 1720 1720 1760 2560 2000 2520 1600 2360 2000 1920 1920 2200 1920 2120 2400 2760 1840 1800 2400 2320 2080 1840 1960 1760 1960 2520 2200 1360 2200 2720 1880 1880 2080 2400 1800 1720 1800 2120 1880 2040 2080 2760 2040 2280 1840 2520 2720 2040 2280 2120 1760 2120 1720 2280 1840 1720 1400 2400
[DEBUG] Parsed TIC intensities for file 20241115_Plasma_TG_22-6.txt: [2040, 1440, 1720, 1720, 1760, 2560, 2000, 2520, 1600, 2360, 2000, 1920, 1920, 2200, 1920, 2120, 2400, 2760, 1840, 1800, 2400, 2320, 2080, 1840, 1960, 1760, 1960, 2520, 2200, 1360, 2200, 2720, 1880, 1880, 2080, 2400, 1800, 1720, 1800, 2120, 1880, 2040, 2080, 2760, 2040, 2280, 1840, 2520, 2720, 2040, 2280, 2120, 1760, 2120, 1720, 2280, 1840, 1720, 1400, 2400]
[DEBUG] Calculated TIC_RSD f

,Date,Sample_Name,Sample,Blank_Group,Lipid,Q1,Q3,Summed_Intensity,TIC_RSD,TIC_Summed,Filename
0,20241115,Plasma_TG_22-6,Sample,1,"""[TG(48:7),TG(47:0)]_FA22:6""",810.76,465.46,2440,16.030326,123440,20241115_Plasma_TG_22
1,20241115,Plasma_TG_22-6,Sample,1,"""[TG(49:7),TG(48:0)]_FA22:6""",824.77,479.47,1560,16.030326,123440,20241115_Plasma_TG_22
2,20241115,Plasma_TG_22-6,Sample,1,"""[TG(50:7),TG(49:0)]_FA22:6""",838.79,493.49,1400,16.030326,123440,20241115_Plasma_TG_22
3,20241115,Plasma_TG_22-6,Sample,1,"""[TG(51:7),TG(50:0)]_FA22:6""",852.80,507.50,1560,16.030326,123440,20241115_Plasma_TG_22
4,20241115,Plasma_TG_22-6,Sample,1,"""[TG(52:7),TG(51:0)]_FA22:6""",866.82,521.52,1080,16.030326,123440,20241115_Plasma_TG_22
...,...,...,...,...,...,...,...,...,...,...,...
227,20241115,Plasma_Blank_TG_22-6,Blank,1,"""[TG(54:11),TG(53:4)]_FA22:6""",886.79,541.49,800,17.995207,103560,20241115_Plasma_Blank_TG_22
228,20241115,Plasma_Blank_TG_22-6,Blank,1,"""[TG(55:10),TG(54:3)]_FA22:6""",902.82,557.52,920,17.995207,103560,20241115_Plasma_Blank_TG_22
229,20241115,Plasma_Blank_TG_22-6,Blank,1,"""[TG(55:11),TG(54:4)]_FA22:6""",900.80,555.50,360,17.995207,103560,20241115_Plasma_Blank_TG_22
230,20241115,Plasma_Blank_TG_22-6,Blank,1,"""[TG(56:9),TG(55:2)]_FA22:6""",918.85,573.55,1160,17.995207,103560,20241115_Plasma_Blank_TG_22


# add blank subtract

In [16]:
import re
import os
import pandas as pd

class QTRAP_Parse:
    def __init__(self, input_dir, output_dir):
        """
        Initialize the QTRAP_Parse class with input and output directories.

        Args:
            input_dir (str): Directory containing the input text files.
            output_dir (str): Directory to save the output CSV file.
        """
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.output_file = os.path.join(self.output_dir, 'parsed_chromatogram_data.csv')

        # Initialize lists to store parsed data
        self.filenames = []
        self.q1_values = []
        self.q3_values = []
        self.lipids = []
        self.dates = []          # List for Date
        self.sample_names = []   # List for Sample_Name
        self.samples = []        # List for Sample
        self.summed_intensities = []  # List for Summed_Intensity
        self.tic_rsd_values = [] # List for TIC_RSD
        self.tic_summed_values = [] # List for TIC_Summed

    def TIC_RSD(self, intensities):
        """
        Calculate the Relative Standard Deviation (RSD) for a given intensity array.

        Args:
            intensities (list of int): The intensity values.

        Returns:
            float: The RSD percentage rounded to two decimal places.
        """
        if not intensities:
            return 0.0
        mean = sum(intensities) / len(intensities)
        if mean == 0:
            return 0.0
        variance = sum((x - mean) ** 2 for x in intensities) / len(intensities)
        std_dev = variance ** 0.5
        rsd = (std_dev / mean) * 100
        return round(rsd, 2)

    def TIC_Summed(self, intensities):
        """
        Calculate the summed Total Ion Current (TIC) for a given intensity array.

        Args:
            intensities (list of int): The intensity values.

        Returns:
            int: The summed TIC.
        """
        return sum(intensities) if intensities else 0

    def parse_file(self, file_name):
        """
        Parse a single chromatogram text file to extract necessary data.

        Args:
            file_name (str): The name of the file to parse.
        """
        file_path = os.path.join(self.input_dir, file_name)

        # Extract Date and Sample_Name from the filename
        base_name = os.path.splitext(file_name)[0]  # Removes the .txt extension
        parts = base_name.split('_', 1)             # Split only on the first underscore
        if len(parts) == 2:
            date_str = parts[0]                     # e.g., '20241115'
            sample_name = parts[1]                  # e.g., 'Plasma_Acyl-Carnitines'
        else:
            # Handle unexpected filename formats
            date_str = ''
            sample_name = ''

        # Determine the Sample value based on Sample_Name
        sample = 'Blank' if 'Blank' in sample_name else 'Sample'

        # Open and read all lines of the file
        try:
            with open(file_path, 'r') as file:
                lines = file.readlines()
        except Exception:
            # Skip the file if it cannot be read
            return

        # -------------------
        # Extract TIC Data
        # -------------------
        TIC_intensities = []
        tic_found = False  # Flag to indicate if TIC section is found
        in_tic_section = False  # Flag to indicate parsing within TIC

        for i, line in enumerate(lines):
            if 'id: TIC' in line:
                tic_found = True
                in_tic_section = True
                continue  # Move to the next line

            if in_tic_section:
                # Look for 'cvParam: intensity array, number of detector counts'
                if 'cvParam: intensity array' in line:
                    # The intensity values are expected in the next line
                    if i + 1 < len(lines):
                        intensity_line = lines[i + 1].strip()
                        match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', intensity_line)
                        if match:
                            try:
                                TIC_intensities = list(map(int, match.group(1).split()))
                            except ValueError:
                                TIC_intensities = []
                        # Exit after finding the intensity array
                    break

        # Calculate TIC_RSD using the TIC_RSD function
        TIC_rsd = self.TIC_RSD(TIC_intensities)

        # Calculate TIC_Summed using the TIC_Summed function
        TIC_sum = self.TIC_Summed(TIC_intensities)

        # -------------------
        # Parse Lipid Data
        # -------------------
        current_filename = ""
        current_q1 = None
        current_q3 = None
        current_lipid = ""
        parsing_intensity = False
        intensities = []

        for line in lines:
            # Extract the filename (assumes filename appears earlier in the file)
            if 'sourceFile:' in line or 'name:' in line:
                match = re.search(r'name:\s+([\w.]+)', line)
                if match:
                    current_filename = match.group(1)

            # Extract Q1, Q3 values, and Lipid name
            if 'id: SRM SIC Q1=' in line:
                match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name=([^\s]+)', line)
                if match:
                    current_q1 = float(match.group(1))
                    current_q3 = float(match.group(2))
                    current_lipid = match.group(3)

            # Check if we are parsing intensity array data for lipids
            if 'cvParam: intensity array' in line:
                parsing_intensity = True
                intensities = []
            elif parsing_intensity and 'binary: [' in line:
                # Extract intensity values
                match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
                if match:
                    try:
                        intensities = list(map(int, match.group(1).split()))
                        current_intensity_sum = sum(intensities)
                    except ValueError:
                        intensities = []
                        current_intensity_sum = 0

                    # Append the extracted and calculated data to the lists
                    if current_filename and current_q1 is not None and current_q3 is not None:
                        self.filenames.append(current_filename)
                        self.q1_values.append(current_q1)
                        self.q3_values.append(current_q3)
                        self.lipids.append(current_lipid)
                        self.dates.append(date_str)              # Append Date
                        self.sample_names.append(sample_name)    # Append Sample_Name
                        self.samples.append(sample)              # Append Sample
                        self.summed_intensities.append(current_intensity_sum)  # Append Summed_Intensity
                        self.tic_rsd_values.append(TIC_rsd)      # Append TIC_RSD
                        self.tic_summed_values.append(TIC_sum)   # Append TIC_Summed

                parsing_intensity = False

    def parse_all_files(self):
        """
        Iterate through all .txt files in the input directory and parse them.
        """
        if not os.path.isdir(self.input_dir):
            return

        # Ensure the output directory exists
        os.makedirs(self.output_dir, exist_ok=True)

        # Iterate through all .txt files in the specified directory
        for file_name in os.listdir(self.input_dir):
            if file_name.endswith('.txt'):
                self.parse_file(file_name)

    def generate_dataframe(self):
        """
        Generate a Pandas DataFrame from the parsed data.

        Returns:
            pd.DataFrame: DataFrame containing the parsed chromatogram data.
        """
        chromatogram_df = pd.DataFrame({
            'Date': self.dates,                        # Date column
            'Sample_Name': self.sample_names,          # Sample_Name column
            'Sample': self.samples,                    # Sample column
            'Lipid': self.lipids,
            'Q1': self.q1_values,
            'Q3': self.q3_values,
            'Summed_Intensity': self.summed_intensities,  # Summed_Intensity column
            'TIC_RSD': self.tic_rsd_values,            # TIC_RSD column
            'TIC_Summed': self.tic_summed_values,      # TIC_Summed column
            'Filename': self.filenames,
        })

        # Create Base_Sample_Name by removing 'Blank_' prefix if present
        chromatogram_df['Base_Sample_Name'] = chromatogram_df['Sample_Name'].str.replace('Blank_', '', regex=False)

        # Assign group numbers based on Base_Sample_Name
        chromatogram_df['Blank_Group'] = pd.factorize(chromatogram_df['Base_Sample_Name'])[0] + 1  # Start groups at 1

        # Optionally, drop the Base_Sample_Name column if not needed
        chromatogram_df.drop(columns=['Base_Sample_Name'], inplace=True)

        # Optionally, convert Date to datetime format
        # chromatogram_df['Date'] = pd.to_datetime(chromatogram_df['Date'], format='%Y%m%d')

        # Perform Blank Subtraction to create 'Intensity_Blank_Subtracted' column
        chromatogram_df = self.Blank_Subtract(chromatogram_df)

        # Reorder columns for better readability
        columns_order = [
            'Date', 
            'Sample_Name', 
            'Sample', 
            'Blank_Group', 
            'Lipid', 
            'Q1', 
            'Q3', 
            'Summed_Intensity',
            'Intensity_Blank_Subtracted',  # New column added here
            'TIC_RSD',                       # Include TIC_RSD in the order
            'TIC_Summed',                   # Include TIC_Summed in the order
            'Filename'
        ]
        chromatogram_df = chromatogram_df[columns_order]

        return chromatogram_df

    def Blank_Subtract(self, df):
        """
        Subtract the Summed_Intensity of blanks from samples within the same Blank_Group and Lipid.

        Args:
            df (pd.DataFrame): The DataFrame containing chromatogram data.

        Returns:
            pd.DataFrame: The DataFrame with the new 'Intensity_Blank_Subtracted' column.
        """
        # Create a DataFrame with blank intensities
        blanks = df[df['Sample'] == 'Blank'][['Blank_Group', 'Lipid', 'Summed_Intensity']].copy()
        blanks.rename(columns={'Summed_Intensity': 'Blank_Summed_Intensity'}, inplace=True)

        # Merge the blank intensities with the original DataFrame on Blank_Group and Lipid
        df = df.merge(blanks, on=['Blank_Group', 'Lipid'], how='left')

        # Define a function to subtract blank intensity from sample intensity
        def subtract_blank(row):
            if row['Sample'] == 'Sample' and not pd.isna(row['Blank_Summed_Intensity']):
                subtracted = row['Summed_Intensity'] - row['Blank_Summed_Intensity']
                return subtracted if subtracted > 0 else 0
            else:
                return 0  # For blanks or if no corresponding blank found

        # Apply the subtraction
        df['Intensity_Blank_Subtracted'] = df.apply(subtract_blank, axis=1)

        # Optionally, drop the 'Blank_Summed_Intensity' column as it's no longer needed
        df.drop(columns=['Blank_Summed_Intensity'], inplace=True)

        return df

    def save_to_csv(self, dataframe):
        """
        Save the DataFrame to a CSV file.

        Args:
            dataframe (pd.DataFrame): The DataFrame to save.
        """
        try:
            dataframe.to_csv(self.output_file, index=False)
        except Exception as e:
            # Handle exceptions silently or implement alternative handling if needed
            print(f"Error saving CSV: {e}")

    def run(self):
        """
        Execute the parsing process: parse all files, generate DataFrame, and save to CSV.

        Returns:
            pd.DataFrame: The resulting DataFrame after parsing.
        """
        self.parse_all_files()
        df = self.generate_dataframe()
        self.save_to_csv(df)
        return df


# update with Classes

In [31]:
import re
import os
import pandas as pd

class QTRAP_Parse:
    def __init__(self, input_dir, output_dir):
        """
        Initialize the QTRAP_Parse class with input and output directories.

        Args:
            input_dir (str): Directory containing the input text files.
            output_dir (str): Directory to save the output CSV file.
        """
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.output_file = os.path.join(self.output_dir, 'parsed_chromatogram_data.csv')

        # Initialize lists to store parsed data
        self.filenames = []
        self.q1_values = []
        self.q3_values = []
        self.lipids = []
        self.dates = []          # List for Date
        self.sample_names = []   # List for Sample_Name
        self.samples = []        # List for Sample
        self.summed_intensities = []  # List for Summed_Intensity
        self.tic_rsd_values = [] # List for TIC_RSD
        self.tic_summed_values = [] # List for TIC_Summed

        # Define lipid classes and their abbreviations
        self.lipid_classes = {
            'Phosphatidylcholine': 'PC',
            'Phosphatidylethanolamine': 'PE',
            'Phosphatidylserine': 'PS',
            'Phosphatidylinositol': 'PI',
            'Sphingomyelin': 'SM',
            'Ceramide': 'Cer',
            'Triglyceride': 'TG',
            'Diacylglycerol': 'DG',
            'Cholesterol Ester': 'CE',
            'Cardiolipin': 'CL',
            'Acyl Carnitine': 'Car',
            'Fatty Acid': 'FA'
        }

    def TIC_RSD(self, intensities):
        """
        Calculate the Relative Standard Deviation (RSD) for a given intensity array.

        Args:
            intensities (list of int): The intensity values.

        Returns:
            float: The RSD percentage rounded to two decimal places.
        """
        if not intensities:
            return 0.0
        mean = sum(intensities) / len(intensities)
        if mean == 0:
            return 0.0
        variance = sum((x - mean) ** 2 for x in intensities) / len(intensities)
        std_dev = variance ** 0.5
        rsd = (std_dev / mean) * 100
        return round(rsd, 2)

    def TIC_Summed(self, intensities):
        """
        Calculate the summed Total Ion Current (TIC) for a given intensity array.

        Args:
            intensities (list of int): The intensity values.

        Returns:
            int: The summed TIC.
        """
        return sum(intensities) if intensities else 0

    def parse_file(self, file_name):
        """
        Parse a single chromatogram text file to extract necessary data.

        Args:
            file_name (str): The name of the file to parse.
        """
        file_path = os.path.join(self.input_dir, file_name)

        # Extract Date and Sample_Name from the filename
        base_name = os.path.splitext(file_name)[0]  # Removes the .txt extension
        parts = base_name.split('_', 1)             # Split only on the first underscore
        if len(parts) == 2:
            date_str = parts[0]                     # e.g., '20241115'
            sample_name = parts[1]                  # e.g., 'Plasma_Acyl-Carnitines'
        else:
            # Handle unexpected filename formats
            date_str = ''
            sample_name = ''

        # Determine the Sample value based on Sample_Name
        sample = 'Blank' if 'Blank' in sample_name else 'Sample'

        # Open and read all lines of the file
        try:
            with open(file_path, 'r') as file:
                lines = file.readlines()
        except Exception as e:
            print(f"Error reading file {file_name}: {e}")
            # Skip the file if it cannot be read
            return

        # -------------------
        # Extract TIC Data
        # -------------------
        TIC_intensities = []
        tic_found = False  # Flag to indicate if TIC section is found
        in_tic_section = False  # Flag to indicate parsing within TIC

        for i, line in enumerate(lines):
            if 'id: TIC' in line:
                tic_found = True
                in_tic_section = True
                continue  # Move to the next line

            if in_tic_section:
                # Look for 'cvParam: intensity array, number of detector counts'
                if 'cvParam: intensity array' in line:
                    # The intensity values are expected in the next line
                    if i + 1 < len(lines):
                        intensity_line = lines[i + 1].strip()
                        match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', intensity_line)
                        if match:
                            try:
                                TIC_intensities = list(map(int, match.group(1).split()))
                            except ValueError:
                                TIC_intensities = []
                        # Exit after finding the intensity array
                    break

        # Calculate TIC_RSD using the TIC_RSD function
        TIC_rsd = self.TIC_RSD(TIC_intensities)

        # Calculate TIC_Summed using the TIC_Summed function
        TIC_sum = self.TIC_Summed(TIC_intensities)

        # -------------------
        # Parse Lipid Data
        # -------------------
        current_filename = ""
        current_q1 = None
        current_q3 = None
        current_lipid = ""
        parsing_intensity = False
        intensities = []

        for line in lines:
            # Extract the filename (assumes filename appears earlier in the file)
            if 'sourceFile:' in line or 'name:' in line:
                match = re.search(r'name:\s+([\w.]+)', line)
                if match:
                    current_filename = match.group(1)

            # Extract Q1, Q3 values, and Lipid name
            if 'id: SRM SIC Q1=' in line:
                match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name=([^\s]+)', line)
                if match:
                    current_q1 = float(match.group(1))
                    current_q3 = float(match.group(2))
                    current_lipid = match.group(3)

            # Check if we are parsing intensity array data for lipids
            if 'cvParam: intensity array' in line:
                parsing_intensity = True
                intensities = []
            elif parsing_intensity and 'binary: [' in line:
                # Extract intensity values
                match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
                if match:
                    try:
                        intensities = list(map(int, match.group(1).split()))
                        current_intensity_sum = sum(intensities)
                    except ValueError:
                        intensities = []
                        current_intensity_sum = 0

                    # Append the extracted and calculated data to the lists
                    if current_filename and current_q1 is not None and current_q3 is not None:
                        self.filenames.append(current_filename)
                        self.q1_values.append(current_q1)
                        self.q3_values.append(current_q3)
                        self.lipids.append(current_lipid)
                        self.dates.append(date_str)              # Append Date
                        self.sample_names.append(sample_name)    # Append Sample_Name
                        self.samples.append(sample)              # Append Sample
                        self.summed_intensities.append(current_intensity_sum)  # Append Summed_Intensity
                        self.tic_rsd_values.append(TIC_rsd)      # Append TIC_RSD
                        self.tic_summed_values.append(TIC_sum)   # Append TIC_Summed

                parsing_intensity = False

    def parse_all_files(self):
        """
        Iterate through all .txt files in the input directory and parse them.
        """
        if not os.path.isdir(self.input_dir):
            print(f"Input directory {self.input_dir} does not exist.")
            return

        # Ensure the output directory exists
        os.makedirs(self.output_dir, exist_ok=True)

        # Iterate through all .txt files in the specified directory
        for file_name in os.listdir(self.input_dir):
            if file_name.endswith('.txt'):
                self.parse_file(file_name)

    def generate_dataframe(self):
        """
        Generate a Pandas DataFrame from the parsed data.

        Returns:
            pd.DataFrame: DataFrame containing the parsed chromatogram data.
        """
        chromatogram_df = pd.DataFrame({
            'Date': self.dates,                        # Date column
            'Sample_Name': self.sample_names,          # Sample_Name column
            'Sample': self.samples,                    # Sample column
            'Lipid': self.lipids,
            'Q1': self.q1_values,
            'Q3': self.q3_values,
            'Summed_Intensity': self.summed_intensities,  # Summed_Intensity column
            'TIC_RSD': self.tic_rsd_values,            # TIC_RSD column
            'TIC_Summed': self.tic_summed_values,      # TIC_Summed column
            'Filename': self.filenames,
        })

        # Create Base_Sample_Name by removing 'Blank_' prefix if present
        chromatogram_df['Base_Sample_Name'] = chromatogram_df['Sample_Name'].str.replace('Blank_', '', regex=False)

        # Assign group numbers based on Base_Sample_Name
        chromatogram_df['Blank_Group'] = pd.factorize(chromatogram_df['Base_Sample_Name'])[0] + 1  # Start groups at 1

        # Drop the Base_Sample_Name column if not needed
        chromatogram_df.drop(columns=['Base_Sample_Name'], inplace=True)

        # Perform Blank Subtraction to create 'Intensity_Blank_Subtracted' column
        chromatogram_df = self.Blank_Subtract(chromatogram_df)

        # Assign Lipid Class Abbreviation using the updated Class function
        chromatogram_df = self.Class(chromatogram_df)

        # Reorder columns for better readability
        columns_order = [
            'Date', 
            'Sample_Name', 
            'Sample', 
            'Blank_Group', 
            'Lipid', 
            'Class',                     # New Class column added here
            'Q1', 
            'Q3', 
            'Summed_Intensity',
            'Intensity_Blank_Subtracted',  # New column added here
            'TIC_RSD',                       # Include TIC_RSD in the order
            'TIC_Summed',                   # Include TIC_Summed in the order
            'Filename'
        ]
        chromatogram_df = chromatogram_df[columns_order]

        return chromatogram_df

    def Blank_Subtract(self, df):
        """
        Subtract the Summed_Intensity of blanks from samples within the same Blank_Group and Lipid.

        Args:
            df (pd.DataFrame): The DataFrame containing chromatogram data.

        Returns:
            pd.DataFrame: The DataFrame with the new 'Intensity_Blank_Subtracted' column.
        """
        # Create a DataFrame with blank intensities
        blanks = df[df['Sample'] == 'Blank'][['Blank_Group', 'Lipid', 'Summed_Intensity']].copy()
        blanks.rename(columns={'Summed_Intensity': 'Blank_Summed_Intensity'}, inplace=True)

        # Merge the blank intensities with the original DataFrame on Blank_Group and Lipid
        df = df.merge(blanks, on=['Blank_Group', 'Lipid'], how='left')

        # Define a function to subtract blank intensity from sample intensity
        def subtract_blank(row):
            if row['Sample'] == 'Sample' and not pd.isna(row['Blank_Summed_Intensity']):
                subtracted = row['Summed_Intensity'] - row['Blank_Summed_Intensity']
                return subtracted if subtracted > 0 else 0
            else:
                return 0  # For blanks or if no corresponding blank found

        # Apply the subtraction
        df['Intensity_Blank_Subtracted'] = df.apply(subtract_blank, axis=1)

        # Drop the 'Blank_Summed_Intensity' column as it's no longer needed
        df.drop(columns=['Blank_Summed_Intensity'], inplace=True)

        return df

    def Class(self, df):
        """
        Assign the Lipid Class Abbreviation based on exact substring matches within Sample_Name.

        Args:
            df (pd.DataFrame): The DataFrame containing chromatogram data.

        Returns:
            pd.DataFrame: The DataFrame with the new 'Class' column.
        """
        # Create a list of lipid abbreviations
        lipid_abbr = list(self.lipid_classes.values())

        # Define a function to assign class based on exact substring match
        def find_class(sample_name):
            for abbr in lipid_abbr:
                # Check if the abbreviation exists as a separate word or delimited by non-alphanumeric characters
                # This ensures exact matches and avoids partial matches
                if re.search(rf'\b{re.escape(abbr)}\b', sample_name):
                    return abbr
            return 'Unknown'

        # Apply the matching function to the Sample_Name column
        df['Class'] = df['Lipid'].apply(find_class)

        return df

    def save_to_csv(self, dataframe):
        """
        Save the DataFrame to a CSV file.

        Args:
            dataframe (pd.DataFrame): The DataFrame to save.
        """
        try:
            dataframe.to_csv(self.output_file, index=False)
            print(f"Data successfully saved to {self.output_file}")
        except Exception as e:
            # Handle exceptions by printing the error
            print(f"Error saving CSV: {e}")

    def run(self):
        """
        Execute the parsing process: parse all files, generate DataFrame, and save to CSV.

        Returns:
            pd.DataFrame: The resulting DataFrame after parsing.
        """
        self.parse_all_files()
        df = self.generate_dataframe()
        self.save_to_csv(df)
        return df


# Normalize ot class

In [33]:
import re
import os
import pandas as pd

class QTRAP_Parse:
    def __init__(self, input_dir, output_dir):
        """
        Initialize the QTRAP_Parse class with input and output directories.

        Args:
            input_dir (str): Directory containing the input text files.
            output_dir (str): Directory to save the output CSV file.
        """
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.output_file = os.path.join(self.output_dir, 'parsed_chromatogram_data.csv')

        # Initialize lists to store parsed data
        self.filenames = []
        self.q1_values = []
        self.q3_values = []
        self.lipids = []
        self.dates = []          # List for Date
        self.sample_names = []   # List for Sample_Name
        self.samples = []        # List for Sample
        self.summed_intensities = []  # List for Summed_Intensity
        self.tic_rsd_values = [] # List for TIC_RSD
        self.tic_summed_values = [] # List for TIC_Summed

        # Define lipid classes and their abbreviations
        self.lipid_classes = {
            'Phosphatidylcholine': 'PC',
            'Phosphatidylethanolamine': 'PE',
            'Phosphatidylserine': 'PS',
            'Phosphatidylinositol': 'PI',
            'Sphingomyelin': 'SM',
            'Ceramide': 'Cer',
            'Triglyceride': 'TG',
            'Diacylglycerol': 'DG',
            'Cholesterol Ester': 'CE',
            'Cardiolipin': 'CL',
            'Acyl Carnitine': 'Car',
            'Fatty Acid': 'FA'
        }

    def TIC_RSD(self, intensities):
        """
        Calculate the Relative Standard Deviation (RSD) for a given intensity array.

        Args:
            intensities (list of int): The intensity values.

        Returns:
            float: The RSD percentage rounded to two decimal places.
        """
        if not intensities:
            return 0.0
        mean = sum(intensities) / len(intensities)
        if mean == 0:
            return 0.0
        variance = sum((x - mean) ** 2 for x in intensities) / len(intensities)
        std_dev = variance ** 0.5
        rsd = (std_dev / mean) * 100
        return round(rsd, 2)

    def TIC_Summed(self, intensities):
        """
        Calculate the summed Total Ion Current (TIC) for a given intensity array.

        Args:
            intensities (list of int): The intensity values.

        Returns:
            int: The summed TIC.
        """
        return sum(intensities) if intensities else 0

    def parse_file(self, file_name):
        """
        Parse a single chromatogram text file to extract necessary data.

        Args:
            file_name (str): The name of the file to parse.
        """
        file_path = os.path.join(self.input_dir, file_name)

        # Extract Date and Sample_Name from the filename
        base_name = os.path.splitext(file_name)[0]  # Removes the .txt extension
        parts = base_name.split('_', 1)             # Split only on the first underscore
        if len(parts) == 2:
            date_str = parts[0]                     # e.g., '20241115'
            sample_name = parts[1]                  # e.g., 'Plasma_Acyl-Carnitines'
        else:
            # Handle unexpected filename formats
            date_str = ''
            sample_name = ''

        # Determine the Sample value based on Sample_Name
        sample = 'Blank' if 'Blank' in sample_name else 'Sample'

        # Open and read all lines of the file
        try:
            with open(file_path, 'r') as file:
                lines = file.readlines()
        except Exception as e:
            print(f"Error reading file {file_name}: {e}")
            # Skip the file if it cannot be read
            return

        # -------------------
        # Extract TIC Data
        # -------------------
        TIC_intensities = []
        tic_found = False  # Flag to indicate if TIC section is found
        in_tic_section = False  # Flag to indicate parsing within TIC

        for i, line in enumerate(lines):
            if 'id: TIC' in line:
                tic_found = True
                in_tic_section = True
                continue  # Move to the next line

            if in_tic_section:
                # Look for 'cvParam: intensity array, number of detector counts'
                if 'cvParam: intensity array' in line:
                    # The intensity values are expected in the next line
                    if i + 1 < len(lines):
                        intensity_line = lines[i + 1].strip()
                        match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', intensity_line)
                        if match:
                            try:
                                TIC_intensities = list(map(int, match.group(1).split()))
                            except ValueError:
                                TIC_intensities = []
                        # Exit after finding the intensity array
                    break

        # Calculate TIC_RSD using the TIC_RSD function
        TIC_rsd = self.TIC_RSD(TIC_intensities)

        # Calculate TIC_Summed using the TIC_Summed function
        TIC_sum = self.TIC_Summed(TIC_intensities)

        # -------------------
        # Parse Lipid Data
        # -------------------
        current_filename = ""
        current_q1 = None
        current_q3 = None
        current_lipid = ""
        parsing_intensity = False
        intensities = []

        for line in lines:
            # Extract the filename (assumes filename appears earlier in the file)
            if 'sourceFile:' in line or 'name:' in line:
                match = re.search(r'name:\s+([\w.]+)', line)
                if match:
                    current_filename = match.group(1)

            # Extract Q1, Q3 values, and Lipid name
            if 'id: SRM SIC Q1=' in line:
                match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name=([^\s]+)', line)
                if match:
                    current_q1 = float(match.group(1))
                    current_q3 = float(match.group(2))
                    current_lipid = match.group(3)

            # Check if we are parsing intensity array data for lipids
            if 'cvParam: intensity array' in line:
                parsing_intensity = True
                intensities = []
            elif parsing_intensity and 'binary: [' in line:
                # Extract intensity values
                match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
                if match:
                    try:
                        intensities = list(map(int, match.group(1).split()))
                        current_intensity_sum = sum(intensities)
                    except ValueError:
                        intensities = []
                        current_intensity_sum = 0

                    # Append the extracted and calculated data to the lists
                    if current_filename and current_q1 is not None and current_q3 is not None:
                        self.filenames.append(current_filename)
                        self.q1_values.append(current_q1)
                        self.q3_values.append(current_q3)
                        self.lipids.append(current_lipid)
                        self.dates.append(date_str)              # Append Date
                        self.sample_names.append(sample_name)    # Append Sample_Name
                        self.samples.append(sample)              # Append Sample
                        self.summed_intensities.append(current_intensity_sum)  # Append Summed_Intensity
                        self.tic_rsd_values.append(TIC_rsd)      # Append TIC_RSD
                        self.tic_summed_values.append(TIC_sum)   # Append TIC_Summed

                parsing_intensity = False

    def parse_all_files(self):
        """
        Iterate through all .txt files in the input directory and parse them.
        """
        if not os.path.isdir(self.input_dir):
            print(f"Input directory {self.input_dir} does not exist.")
            return

        # Ensure the output directory exists
        os.makedirs(self.output_dir, exist_ok=True)

        # Iterate through all .txt files in the specified directory
        for file_name in os.listdir(self.input_dir):
            if file_name.endswith('.txt'):
                self.parse_file(file_name)

    def generate_dataframe(self):
        """
        Generate a Pandas DataFrame from the parsed data.

        Returns:
            pd.DataFrame: DataFrame containing the parsed chromatogram data.
        """
        chromatogram_df = pd.DataFrame({
            'Date': self.dates,                        # Date column
            'Sample_Name': self.sample_names,          # Sample_Name column
            'Sample': self.samples,                    # Sample column
            'Lipid': self.lipids,
            'Q1': self.q1_values,
            'Q3': self.q3_values,
            'Summed_Intensity': self.summed_intensities,  # Summed_Intensity column
            'TIC_RSD': self.tic_rsd_values,            # TIC_RSD column
            'TIC_Summed': self.tic_summed_values,      # TIC_Summed column
            'Filename': self.filenames,
        })

        # Create Base_Sample_Name by removing 'Blank_' prefix if present
        chromatogram_df['Base_Sample_Name'] = chromatogram_df['Sample_Name'].str.replace('Blank_', '', regex=False)

        # Assign group numbers based on Base_Sample_Name
        chromatogram_df['Blank_Group'] = pd.factorize(chromatogram_df['Base_Sample_Name'])[0] + 1  # Start groups at 1

        # Drop the Base_Sample_Name column if not needed
        chromatogram_df.drop(columns=['Base_Sample_Name'], inplace=True)

        # Perform Blank Subtraction to create 'Intensity_Blank_Subtracted' column
        chromatogram_df = self.Blank_Subtract(chromatogram_df)

        # Assign Lipid Class Abbreviation using the updated Class function
        chromatogram_df = self.Class(chromatogram_df)

        # Reorder columns for better readability
        columns_order = [
            'Date', 
            'Sample_Name', 
            'Sample', 
            'Blank_Group', 
            'Lipid', 
            'Class',                     # New Class column added here
            'Q1', 
            'Q3', 
            'Summed_Intensity',
            'Intensity_Blank_Subtracted',  # New column added here
            'TIC_RSD',                       # Include TIC_RSD in the order
            'TIC_Summed',                   # Include TIC_Summed in the order
            'Filename'
        ]
        chromatogram_df = chromatogram_df[columns_order]

        # Normalize intensities within each Class for each Filename
        chromatogram_df = self.normalize_to_class(chromatogram_df)

        return chromatogram_df

    def Blank_Subtract(self, df):
        """
        Subtract the Summed_Intensity of blanks from samples within the same Blank_Group and Lipid.

        Args:
            df (pd.DataFrame): The DataFrame containing chromatogram data.

        Returns:
            pd.DataFrame: The DataFrame with the new 'Intensity_Blank_Subtracted' column.
        """
        # Create a DataFrame with blank intensities
        blanks = df[df['Sample'] == 'Blank'][['Blank_Group', 'Lipid', 'Summed_Intensity']].copy()
        blanks.rename(columns={'Summed_Intensity': 'Blank_Summed_Intensity'}, inplace=True)

        # Merge the blank intensities with the original DataFrame on Blank_Group and Lipid
        df = df.merge(blanks, on=['Blank_Group', 'Lipid'], how='left')

        # Define a function to subtract blank intensity from sample intensity
        def subtract_blank(row):
            if row['Sample'] == 'Sample' and not pd.isna(row['Blank_Summed_Intensity']):
                subtracted = row['Summed_Intensity'] - row['Blank_Summed_Intensity']
                return subtracted if subtracted > 0 else 0
            else:
                return 0  # For blanks or if no corresponding blank found

        # Apply the subtraction
        df['Intensity_Blank_Subtracted'] = df.apply(subtract_blank, axis=1)

        # Drop the 'Blank_Summed_Intensity' column as it's no longer needed
        df.drop(columns=['Blank_Summed_Intensity'], inplace=True)

        return df

    def Class(self, df):
        """
        Assign the Lipid Class Abbreviation based on exact substring matches within Sample_Name.

        Args:
            df (pd.DataFrame): The DataFrame containing chromatogram data.

        Returns:
            pd.DataFrame: The DataFrame with the new 'Class' column.
        """
        # Create a list of lipid abbreviations
        lipid_abbr = list(self.lipid_classes.values())

        # Define a function to assign class based on exact substring match
        def find_class(lipid):
            for abbr in lipid_abbr:
                # Check if the abbreviation exists as a separate word or delimited by non-alphanumeric characters
                # This ensures exact matches and avoids partial matches
                if re.search(rf'\b{re.escape(abbr)}\b', lipid):
                    return abbr
            return 'Unknown'

        # Apply the matching function to the Lipid column
        df['Class'] = df['Lipid'].apply(find_class)

        return df

    def normalize_to_class(self, df):
        """
        Normalize the 'Intensity_Blank_Subtracted' values to the highest value in each Class for each Filename.
        The highest value in each group is set to 1, and others are scaled accordingly.

        Args:
            df (pd.DataFrame): The DataFrame containing chromatogram data.

        Returns:
            pd.DataFrame: The DataFrame with the new 'Normalized_Intensity' column.
        """
        # Using transform with lambda for efficiency
        df['Normalized_Intensity'] = df.groupby(['Filename', 'Class'])['Intensity_Blank_Subtracted'].transform(
            lambda x: x / x.max() if x.max() > 0 else 0
        )
        return df

    def save_to_csv(self, dataframe):
        """
        Save the DataFrame to a CSV file.

        Args:
            dataframe (pd.DataFrame): The DataFrame to save.
        """
        try:
            dataframe.to_csv(self.output_file, index=False)
            print(f"Data successfully saved to {self.output_file}")
        except Exception as e:
            # Handle exceptions by printing the error
            print(f"Error saving CSV: {e}")

    def run(self):
        """
        Execute the parsing process: parse all files, generate DataFrame, and save to CSV.

        Returns:
            pd.DataFrame: The resulting DataFrame after parsing.
        """
        self.parse_all_files()
        df = self.generate_dataframe()
        self.save_to_csv(df)
        return df


# fix order columns

In [37]:
import re
import os
import pandas as pd

class QTRAP_Parse:
    def __init__(self, input_dir, output_dir):
        """
        Initialize the QTRAP_Parse class with input and output directories.

        Args:
            input_dir (str): Directory containing the input text files.
            output_dir (str): Directory to save the output CSV file.
        """
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.output_file = os.path.join(self.output_dir, 'parsed_chromatogram_data.csv')

        # Initialize lists to store parsed data
        self.filenames = []
        self.q1_values = []
        self.q3_values = []
        self.lipids = []
        self.dates = []          # List for Date
        self.sample_names = []   # List for Sample_Name
        self.samples = []        # List for Sample
        self.summed_intensities = []  # List for Summed_Intensity
        self.tic_rsd_values = [] # List for TIC_RSD
        self.tic_summed_values = [] # List for TIC_Summed

        # Define lipid classes and their abbreviations
        self.lipid_classes = {
            'Phosphatidylcholine': 'PC',
            'Phosphatidylethanolamine': 'PE',
            'Phosphatidylserine': 'PS',
            'Phosphatidylinositol': 'PI',
            'Sphingomyelin': 'SM',
            'Ceramide': 'Cer',
            'Triglyceride': 'TG',
            'Diacylglycerol': 'DG',
            'Cholesterol Ester': 'CE',
            'Cardiolipin': 'CL',
            'Acyl Carnitine': 'Car',
            'Fatty Acid': 'FA'
        }

    def TIC_RSD(self, intensities):
        """
        Calculate the Relative Standard Deviation (RSD) for a given intensity array.

        Args:
            intensities (list of int): The intensity values.

        Returns:
            float: The RSD percentage rounded to two decimal places.
        """
        if not intensities:
            return 0.0
        mean = sum(intensities) / len(intensities)
        if mean == 0:
            return 0.0
        variance = sum((x - mean) ** 2 for x in intensities) / len(intensities)
        std_dev = variance ** 0.5
        rsd = (std_dev / mean) * 100
        return round(rsd, 2)

    def TIC_Summed(self, intensities):
        """
        Calculate the summed Total Ion Current (TIC) for a given intensity array.

        Args:
            intensities (list of int): The intensity values.

        Returns:
            int: The summed TIC.
        """
        return sum(intensities) if intensities else 0

    def parse_file(self, file_name):
        """
        Parse a single chromatogram text file to extract necessary data.

        Args:
            file_name (str): The name of the file to parse.
        """
        file_path = os.path.join(self.input_dir, file_name)

        # Extract Date and Sample_Name from the filename
        base_name = os.path.splitext(file_name)[0]  # Removes the .txt extension
        parts = base_name.split('_', 1)             # Split only on the first underscore
        if len(parts) == 2:
            date_str = parts[0]                     # e.g., '20241115'
            sample_name = parts[1]                  # e.g., 'Plasma_Acyl-Carnitines'
        else:
            # Handle unexpected filename formats
            date_str = ''
            sample_name = ''

        # Determine the Sample value based on Sample_Name
        sample = 'Blank' if 'Blank' in sample_name else 'Sample'

        # Open and read all lines of the file
        try:
            with open(file_path, 'r') as file:
                lines = file.readlines()
        except Exception as e:
            print(f"Error reading file {file_name}: {e}")
            # Skip the file if it cannot be read
            return

        # -------------------
        # Extract TIC Data
        # -------------------
        TIC_intensities = []
        tic_found = False  # Flag to indicate if TIC section is found
        in_tic_section = False  # Flag to indicate parsing within TIC

        for i, line in enumerate(lines):
            if 'id: TIC' in line:
                tic_found = True
                in_tic_section = True
                continue  # Move to the next line

            if in_tic_section:
                # Look for 'cvParam: intensity array, number of detector counts'
                if 'cvParam: intensity array' in line:
                    # The intensity values are expected in the next line
                    if i + 1 < len(lines):
                        intensity_line = lines[i + 1].strip()
                        match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', intensity_line)
                        if match:
                            try:
                                TIC_intensities = list(map(int, match.group(1).split()))
                            except ValueError:
                                TIC_intensities = []
                        # Exit after finding the intensity array
                    break

        # Calculate TIC_RSD using the TIC_RSD function
        TIC_rsd = self.TIC_RSD(TIC_intensities)

        # Calculate TIC_Summed using the TIC_Summed function
        TIC_sum = self.TIC_Summed(TIC_intensities)

        # -------------------
        # Parse Lipid Data
        # -------------------
        current_filename = ""
        current_q1 = None
        current_q3 = None
        current_lipid = ""
        parsing_intensity = False
        intensities = []

        for line in lines:
            # Extract the filename (assumes filename appears earlier in the file)
            if 'sourceFile:' in line or 'name:' in line:
                match = re.search(r'name:\s+([\w.]+)', line)
                if match:
                    current_filename = match.group(1)

            # Extract Q1, Q3 values, and Lipid name
            if 'id: SRM SIC Q1=' in line:
                match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name=([^\s]+)', line)
                if match:
                    current_q1 = float(match.group(1))
                    current_q3 = float(match.group(2))
                    current_lipid = match.group(3)

            # Check if we are parsing intensity array data for lipids
            if 'cvParam: intensity array' in line:
                parsing_intensity = True
                intensities = []
            elif parsing_intensity and 'binary: [' in line:
                # Extract intensity values
                match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
                if match:
                    try:
                        intensities = list(map(int, match.group(1).split()))
                        current_intensity_sum = sum(intensities)
                    except ValueError:
                        intensities = []
                        current_intensity_sum = 0

                    # Append the extracted and calculated data to the lists
                    if current_filename and current_q1 is not None and current_q3 is not None:
                        self.filenames.append(current_filename)
                        self.q1_values.append(current_q1)
                        self.q3_values.append(current_q3)
                        self.lipids.append(current_lipid)
                        self.dates.append(date_str)              # Append Date
                        self.sample_names.append(sample_name)    # Append Sample_Name
                        self.samples.append(sample)              # Append Sample
                        self.summed_intensities.append(current_intensity_sum)  # Append Summed_Intensity
                        self.tic_rsd_values.append(TIC_rsd)      # Append TIC_RSD
                        self.tic_summed_values.append(TIC_sum)   # Append TIC_Summed

                parsing_intensity = False

    def parse_all_files(self):
        """
        Iterate through all .txt files in the input directory and parse them.
        """
        if not os.path.isdir(self.input_dir):
            print(f"Input directory {self.input_dir} does not exist.")
            return

        # Ensure the output directory exists
        os.makedirs(self.output_dir, exist_ok=True)

        # Iterate through all .txt files in the specified directory
        for file_name in os.listdir(self.input_dir):
            if file_name.endswith('.txt'):
                self.parse_file(file_name)

    def generate_dataframe(self):
        """
        Generate a Pandas DataFrame from the parsed data.

        Returns:
            pd.DataFrame: DataFrame containing the parsed chromatogram data.
        """
        chromatogram_df = pd.DataFrame({
            'Date': self.dates,                        # Date column
            'Sample_Name': self.sample_names,          # Sample_Name column
            'Sample': self.samples,                    # Sample column
            'Lipid': self.lipids,
            'Q1': self.q1_values,
            'Q3': self.q3_values,
            'Summed_Intensity': self.summed_intensities,  # Summed_Intensity column
            'TIC_RSD': self.tic_rsd_values,            # TIC_RSD column
            'TIC_Summed': self.tic_summed_values,      # TIC_Summed column
            'Filename': self.filenames,
        })

        # Create Base_Sample_Name by removing 'Blank_' prefix if present
        chromatogram_df['Base_Sample_Name'] = chromatogram_df['Sample_Name'].str.replace('Blank_', '', regex=False)

        # Assign group numbers based on Base_Sample_Name
        chromatogram_df['Blank_Group'] = pd.factorize(chromatogram_df['Base_Sample_Name'])[0] + 1  # Start groups at 1

        # Drop the Base_Sample_Name column if not needed
        chromatogram_df.drop(columns=['Base_Sample_Name'], inplace=True)

        # Perform Blank Subtraction to create 'Intensity_Blank_Subtracted' column
        chromatogram_df = self.Blank_Subtract(chromatogram_df)

        # Assign Lipid Class Abbreviation using the updated Class function
        chromatogram_df = self.Class(chromatogram_df)

        # Normalize intensities within each Class for each Filename
        chromatogram_df = self.normalize_to_class(chromatogram_df)

        # Reorder columns for better readability as per user request
        columns_order = [
            'Lipid',
            'Q1',
            'Q3',
            'Summed_Intensity',
            'Intensity_Blank_Subtracted',
            'Normalized_Intensity',
            'TIC_RSD',
            'Class',
            'Sample_Name',
            'Blank_Group',
            # Include any additional columns if necessary
            'Date',
            'Sample',
            'TIC_Summed',
            'Filename'
        ]

        # Check if 'Normalized_Intensity' exists; if not, create it with default value 0
        if 'Normalized_Intensity' not in chromatogram_df.columns:
            chromatogram_df['Normalized_Intensity'] = 0

        # Reorder the columns
        chromatogram_df = chromatogram_df[columns_order]

        return chromatogram_df

    def Blank_Subtract(self, df):
        """
        Subtract the Summed_Intensity of blanks from samples within the same Blank_Group and Lipid.

        Args:
            df (pd.DataFrame): The DataFrame containing chromatogram data.

        Returns:
            pd.DataFrame: The DataFrame with the new 'Intensity_Blank_Subtracted' column.
        """
        # Create a DataFrame with blank intensities
        blanks = df[df['Sample'] == 'Blank'][['Blank_Group', 'Lipid', 'Summed_Intensity']].copy()
        blanks.rename(columns={'Summed_Intensity': 'Blank_Summed_Intensity'}, inplace=True)

        # Merge the blank intensities with the original DataFrame on Blank_Group and Lipid
        df = df.merge(blanks, on=['Blank_Group', 'Lipid'], how='left')

        # Define a function to subtract blank intensity from sample intensity
        def subtract_blank(row):
            if row['Sample'] == 'Sample' and not pd.isna(row['Blank_Summed_Intensity']):
                subtracted = row['Summed_Intensity'] - row['Blank_Summed_Intensity']
                return subtracted if subtracted > 0 else 0
            else:
                return 0  # For blanks or if no corresponding blank found

        # Apply the subtraction
        df['Intensity_Blank_Subtracted'] = df.apply(subtract_blank, axis=1)

        # Drop the 'Blank_Summed_Intensity' column as it's no longer needed
        df.drop(columns=['Blank_Summed_Intensity'], inplace=True)

        return df

    def Class(self, df):
        """
        Assign the Lipid Class Abbreviation based on exact substring matches within Lipid.

        Args:
            df (pd.DataFrame): The DataFrame containing chromatogram data.

        Returns:
            pd.DataFrame: The DataFrame with the new 'Class' column.
        """
        # Create a list of lipid abbreviations
        lipid_abbr = list(self.lipid_classes.values())

        # Define a function to assign class based on exact substring match
        def find_class(lipid):
            for abbr in lipid_abbr:
                # Check if the abbreviation exists as a separate word or delimited by non-alphanumeric characters
                # This ensures exact matches and avoids partial matches
                if re.search(rf'\b{re.escape(abbr)}\b', lipid):
                    return abbr
            return 'Unknown'

        # Apply the matching function to the Lipid column
        df['Class'] = df['Lipid'].apply(find_class)

        return df

    def normalize_to_class(self, df):
        """
        Normalize the 'Intensity_Blank_Subtracted' values to the highest value in each Class for each Filename.
        The highest value in each group is set to 1, and others are scaled accordingly.

        Args:
            df (pd.DataFrame): The DataFrame containing chromatogram data.

        Returns:
            pd.DataFrame: The DataFrame with the new 'Normalized_Intensity' column.
        """
        # Using transform with lambda for efficiency
        df['Normalized_Intensity'] = df.groupby(['Filename', 'Class'])['Intensity_Blank_Subtracted'].transform(
            lambda x: x / x.max() if x.max() > 0 else 0
        )
        return df

    def save_to_csv(self, dataframe):
        """
        Save the DataFrame to a CSV file.

        Args:
            dataframe (pd.DataFrame): The DataFrame to save.
        """
        try:
            dataframe.to_csv(self.output_file, index=False)
            print(f"Data successfully saved to {self.output_file}")
        except Exception as e:
            # Handle exceptions by printing the error
            print(f"Error saving CSV: {e}")

    def run(self):
        """
        Execute the parsing process: parse all files, generate DataFrame, and save to CSV.

        Returns:
            pd.DataFrame: The resulting DataFrame after parsing.
        """
        self.parse_all_files()
        df = self.generate_dataframe()
        self.save_to_csv(df)
        return df


In [38]:
# Example usage of the QTRAP_Parse class

# Define input and output directories
input_dir = 'data/text/'
output_dir = 'result/'
# Initialize the parser
parser = QTRAP_Parse(input_dir=input_dir, output_dir=output_dir)

# Run the parsing process
chromatogram_data = parser.run()

# Display the first few rows of the resulting DataFrame
# sort by Intensity_Blank_Subtracted in descending order
chromatogram_data.sort_values(by='Intensity_Blank_Subtracted', ascending=False).head(20)
#chromatogram_data


Data successfully saved to result/parsed_chromatogram_data.csv


,Lipid,Q1,Q3,Summed_Intensity,Intensity_Blank_Subtracted,Normalized_Intensity,TIC_RSD,Class,Sample_Name,Blank_Group,Date,Sample,TIC_Summed,Filename
55,"""[TG(53:9),TG(52:2)]_FA22:6""",876.80,531.50,9520,8520,1.000000,16.03,TG,Plasma_TG_22-6,1,20241115,Sample,123440,20241115_Plasma_TG_22
23,"""[TG(61:12),TG(60:5)]_FA22:6""",982.88,637.58,12360,7480,0.877934,16.03,TG,Plasma_TG_22-6,1,20241115,Sample,123440,20241115_Plasma_TG_22
98,"""[TG(63:13),TG(62:6)]_FA22:6""",1008.90,663.60,5000,2400,0.281690,16.03,TG,Plasma_TG_22-6,1,20241115,Sample,123440,20241115_Plasma_TG_22
71,"""[TG(53:8),TG(52:1)]_FA22:6""",878.82,533.52,2360,1320,0.154930,16.03,TG,Plasma_TG_22-6,1,20241115,Sample,123440,20241115_Plasma_TG_22
111,"""[TG(54:11),TG(53:4)]_FA22:6""",886.79,541.49,1800,1000,0.117371,16.03,TG,Plasma_TG_22-6,1,20241115,Sample,123440,20241115_Plasma_TG_22
62,"""[TG(64:16),TG(63:9),TG(62:2)]_FA22:6""",1016.96,671.66,1280,840,0.098592,16.03,TG,Plasma_TG_22-6,1,20241115,Sample,123440,20241115_Plasma_TG_22
106,[TG(54:6)]_FA22:6,896.77,551.47,1240,840,0.098592,16.03,TG,Plasma_TG_22-6,1,20241115,Sample,123440,20241115_Plasma_TG_22
95,"""[TG(60:13),TG(59:6)]_FA22:6""",966.85,621.55,1480,640,0.075117,16.03,TG,Plasma_TG_22-6,1,20241115,Sample,123440,20241115_Plasma_TG_22
92,[TG(56:6)]_FA22:6,924.80,579.50,800,600,0.070423,16.03,TG,Plasma_TG_22-6,1,20241115,Sample,123440,20241115_Plasma_TG_22
32,"""[TG(59:11),TG(58:4)]_FA22:6""",956.86,611.56,1040,600,0.070423,16.03,TG,Plasma_TG_22-6,1,20241115,Sample,123440,20241115_Plasma_TG_22
